# Health Insurance Cross Sell Prediction: Data Extraction & Database Setup

**Scope of this notebook:** structuring the Kaggle dataset, extracting/loading the CSV, and loading it into MySQL/MariaDB (via XAMPP). No machine learning yet.

**Prerequisites:**
- **XAMPP** — open the XAMPP Control Panel and click **Start** next to *MySQL*. Wait until the status turns green (this is MariaDB, MySQL-compatible). Already confirmed running on port 3306 for this project.
- **DBeaver** — create a new connection: Database menu -> New Database Connection -> MySQL. Host `localhost`, Port `3306`, Username `root`, Password *(blank, XAMPP default)*. Test Connection, then Finish.
- The `health_insurance_db` database and `customers` table already exist (created by this notebook the first time it ran). The cells below recreate them with `IF NOT EXISTS`, so re-running this notebook is always safe and never creates a duplicate database.

Dataset files expected in `../data/`: `train.csv`, `test.csv`, `sample_submission.csv` (from the Kaggle "Health Insurance Cross Sell Prediction" dataset).

## Import libraries

`pandas` reads/structures the CSV. `sqlalchemy` + `pymysql` connect Python to MySQL/MariaDB. If this cell fails with `ModuleNotFoundError`, run in a terminal: `pip install pandas sqlalchemy pymysql`.

In [1]:
# Environment check: makes sure this kernel has every package the notebook needs before anything else runs.
# If it fails, either pick the kernel "Health Insurance (.venv)" (top-right kernel picker) or install the
# missing packages into the selected Python:  python -m pip install -r ../app/requirements.txt
import importlib.util
import sys

required = {'pandas': 'pandas', 'numpy': 'numpy', 'sqlalchemy': 'SQLAlchemy', 'pymysql': 'PyMySQL',
            'sklearn': 'scikit-learn', 'matplotlib': 'matplotlib', 'seaborn': 'seaborn', 'scipy': 'scipy', 'joblib': 'joblib'}
missing = [pip_name for module, pip_name in required.items() if importlib.util.find_spec(module) is None]
if missing:
    raise ModuleNotFoundError(
        f'This kernel ({sys.executable}) is missing: {", ".join(missing)}. '
        'Fix: select the kernel "Health Insurance (.venv)" in the kernel picker (top-right), '
        f'or run:  "{sys.executable}" -m pip install {" ".join(missing)}'
    )
print(f'Environment OK: {sys.executable}')

Environment OK: D:\Practice\.venv\Scripts\python.exe


In [2]:
import pandas as pd
from sqlalchemy import create_engine, text

pd.set_option('display.max_columns', None)
print('pandas version:', pd.__version__)

pandas version: 3.0.5


## 1. Load the Kaggle CSV dataset

Loads `train.csv` — the file that contains all 12 columns including the target column `Response`. (`test.csv` has the same columns minus `Response`, and `sample_submission.csv` is just the submission format for Kaggle's competition — neither is needed for the database step.)

`CSV_PATH` is resolved from the project root, so it works whether the kernel starts in `notebook/` or in the project folder.

In [3]:
from pathlib import Path

# Resolve data/ from the project root so this works whether the kernel started in notebook/
# (the Jupyter / VS Code default) or in the project folder itself.
working_directory = Path.cwd()
PROJECT_ROOT = working_directory.parent if working_directory.name == 'notebook' else working_directory
CSV_PATH = PROJECT_ROOT / 'data' / 'train.csv'

df = pd.read_csv(CSV_PATH)
print(f'Loaded {len(df):,} rows and {df.shape[1]} columns from {CSV_PATH}')
df.head()

Loaded 381,109 rows and 12 columns from D:\Practice\data\train.csv


,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage,Response
0,1,Male,44,1,28.0,0,> 2 Years,Yes,40454.0,26.0,217,1
1,2,Male,76,1,3.0,0,1-2 Year,No,33536.0,26.0,183,0
2,3,Male,47,1,28.0,0,> 2 Years,Yes,38294.0,26.0,27,1
3,4,Male,21,1,11.0,1,< 1 Year,No,28619.0,152.0,203,0
4,5,Female,29,1,41.0,1,< 1 Year,No,27496.0,152.0,39,0


## 2. Dataset structure

Looks at the shape, column names, and a `.info()` summary so we understand what we are about to put into a database table.

In [4]:
print('Shape (rows, columns):', df.shape)
print('\nColumns:', list(df.columns))
print()
df.info()

Shape (rows, columns): (381109, 12)

Columns: ['id', 'Gender', 'Age', 'Driving_License', 'Region_Code', 'Previously_Insured', 'Vehicle_Age', 'Vehicle_Damage', 'Annual_Premium', 'Policy_Sales_Channel', 'Vintage', 'Response']



<class 'pandas.DataFrame'>
RangeIndex: 381109 entries, 0 to 381108
Data columns (total 12 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    381109 non-null  int64  
 1   Gender                381109 non-null  str    
 2   Age                   381109 non-null  int64  
 3   Driving_License       381109 non-null  int64  
 4   Region_Code           381109 non-null  float64
 5   Previously_Insured    381109 non-null  int64  
 6   Vehicle_Age           381109 non-null  str    
 7   Vehicle_Damage        381109 non-null  str    
 8   Annual_Premium        381109 non-null  float64
 9   Policy_Sales_Channel  381109 non-null  float64
 10  Vintage               381109 non-null  int64  
 11  Response              381109 non-null  int64  
dtypes: float64(3), int64(6), str(3)
memory usage: 40.5 MB


In [5]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id,381109.0,NaN,NaN,NaN,190555.0,110016.836208,1.0,95278.0,190555.0,285832.0,381109.0
Gender,381109,2,Male,206089,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age,381109.0,NaN,NaN,NaN,38.822584,15.511611,20.0,25.0,36.0,49.0,85.0
Driving_License,381109.0,NaN,NaN,NaN,0.997869,0.04611,0.0,1.0,1.0,1.0,1.0
Region_Code,381109.0,NaN,NaN,NaN,26.388807,13.229888,0.0,15.0,28.0,35.0,52.0
Previously_Insured,381109.0,NaN,NaN,NaN,0.45821,0.498251,0.0,0.0,0.0,1.0,1.0
Vehicle_Age,381109,3,1-2 Year,200316,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Vehicle_Damage,381109,2,Yes,192413,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Annual_Premium,381109.0,NaN,NaN,NaN,30564.389581,17213.155057,2630.0,24405.0,31669.0,39400.0,540165.0
Policy_Sales_Channel,381109.0,NaN,NaN,NaN,112.034295,54.203995,1.0,29.0,133.0,152.0,163.0


## 3. Data types, missing values, and duplicates

The Kaggle train set is clean (0 nulls, 0 duplicate rows, `id` is unique), but we check anyway rather than assuming — this is what decides whether the MySQL columns can be `NOT NULL` and whether `id` is safe as a `PRIMARY KEY`.

In [6]:
print('Data types:')
print(df.dtypes)

print('\nMissing values per column:')
print(df.isnull().sum())

print('\nFully duplicated rows:', df.duplicated().sum())
print('Duplicate ids:', df['id'].duplicated().sum())
print('id is unique:', df['id'].is_unique)

print('\nUnique values in categorical columns:')
for col in ['Gender', 'Vehicle_Age', 'Vehicle_Damage']:
    print(f'  {col}: {sorted(df[col].unique())}')

Data types:
id                        int64
Gender                      str
Age                       int64
Driving_License           int64
Region_Code             float64
Previously_Insured        int64
Vehicle_Age                 str
Vehicle_Damage              str
Annual_Premium          float64
Policy_Sales_Channel    float64
Vintage                   int64
Response                  int64
dtype: object

Missing values per column:
id                      0
Gender                  0
Age                     0
Driving_License         0
Region_Code             0
Previously_Insured      0
Vehicle_Age             0
Vehicle_Damage          0
Annual_Premium          0
Policy_Sales_Channel    0
Vintage                 0
Response                0
dtype: int64



Fully duplicated rows: 0
Duplicate ids: 0
id is unique: True

Unique values in categorical columns:
  Gender: ['Female', 'Male']
  Vehicle_Age: ['1-2 Year', '< 1 Year', '> 2 Years']
  Vehicle_Damage: ['No', 'Yes']


## 4. Connect to MySQL/MariaDB (XAMPP)

MySQL must already be **started in XAMPP** (Control Panel -> MySQL -> Start) before this cell will succeed.

This first connection does **not** specify a database yet — we only need server access to create `health_insurance_db` next. Default XAMPP credentials are `root` with **no password**; change `DB_USER`/`DB_PASSWORD` below if you've customized yours.

In [7]:
DB_HOST = 'localhost'
DB_PORT = 3306
DB_USER = 'root'
DB_PASSWORD = ''  # XAMPP default: empty password
DB_NAME = 'health_insurance_db'

# Connection WITHOUT a database selected (server-level), used only to CREATE DATABASE
server_engine = create_engine(f'mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/')

with server_engine.connect() as conn:
    version = conn.execute(text('SELECT VERSION()')).scalar()
    print('Connected successfully. Server version:', version)

Connected successfully. Server version: 10.4.32-MariaDB


## 5. Create the `health_insurance_db` database

`CREATE DATABASE IF NOT EXISTS` — safe to run even though the database already exists; it's a no-op in that case, not a duplicate.

In [8]:
with server_engine.connect() as conn:
    conn.execute(text(
        f"CREATE DATABASE IF NOT EXISTS {DB_NAME} "
        "CHARACTER SET utf8mb4 COLLATE utf8mb4_general_ci"
    ))
    conn.commit()
print(f'Database "{DB_NAME}" is ready.')

# Now connect WITH the database selected — this engine is used for the rest of the notebook
engine = create_engine(f'mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}')

Database "health_insurance_db" is ready.


## 6. Create the customers table

Creates the `customers` table with column types chosen to match the real data (checked earlier): `id` unique -> `PRIMARY KEY`; `Gender`/`Vehicle_Age`/`Vehicle_Damage` have a fixed small set of values -> `ENUM`; 0/1 flag columns -> `TINYINT(1)`. `IF NOT EXISTS` means this is a no-op against the existing table, not a re-creation.

In [9]:
CREATE_TABLE_SQL = """
CREATE TABLE IF NOT EXISTS customers (
    id                     INT UNSIGNED PRIMARY KEY,
    gender                 ENUM('Male', 'Female') NOT NULL,
    age                    TINYINT UNSIGNED NOT NULL,
    driving_license        TINYINT(1) NOT NULL,
    region_code            SMALLINT UNSIGNED NOT NULL,
    previously_insured     TINYINT(1) NOT NULL,
    vehicle_age            ENUM('< 1 Year', '1-2 Year', '> 2 Years') NOT NULL,
    vehicle_damage         ENUM('Yes', 'No') NOT NULL,
    annual_premium         DECIMAL(10, 2) NOT NULL,
    policy_sales_channel   SMALLINT UNSIGNED NOT NULL,
    vintage                SMALLINT UNSIGNED NOT NULL,
    response               TINYINT(1) NOT NULL
) ENGINE=InnoDB
"""

with engine.connect() as conn:
    conn.execute(text(CREATE_TABLE_SQL))
    conn.commit()
print('Table "customers" is ready.')

Table "customers" is ready.


## 7. Load the CSV data into MySQL

The MySQL table uses lowercase `snake_case` column names, so we rename the DataFrame columns to match before inserting. The old rows are deleted first, in the same transaction as the insert, so re-running this cell never fails on duplicate primary keys and always leaves the table matching the current CSV. If the cell is stopped or fails part-way, the whole load is rolled back and the previous table stays intact (never a partial table). `to_sql(..., if_exists='append')` loads the ~381k rows in batches and prints progress; expect roughly 30-60 seconds.

In [20]:
COLUMN_MAP = {
    'id': 'id',
    'Gender': 'gender',
    'Age': 'age',
    'Driving_License': 'driving_license',
    'Region_Code': 'region_code',
    'Previously_Insured': 'previously_insured',
    'Vehicle_Age': 'vehicle_age',
    'Vehicle_Damage': 'vehicle_damage',
    'Annual_Premium': 'annual_premium',
    'Policy_Sales_Channel': 'policy_sales_channel',
    'Vintage': 'vintage',
    'Response': 'response',
}

df_sql = df.rename(columns=COLUMN_MAP).copy()

# Region_Code / Policy_Sales_Channel arrive as floats (e.g. 28.0) but are whole-number codes
df_sql['region_code'] = df_sql['region_code'].astype(int)
df_sql['policy_sales_channel'] = df_sql['policy_sales_channel'].astype(int)

# Empty the table and load the rows in ONE transaction: if the insert fails or is stopped part-way,
# MySQL rolls it all back and the previous full table stays. (TRUNCATE commits immediately, and
# to_sql(con=engine) commits the chunks already inserted when interrupted -- together they left a
# partial table.) pandas' default executemany insert is ~3x faster here than method='multi'.
BATCH_ROWS = 50_000
with engine.begin() as conn:
    conn.execute(text('DELETE FROM customers'))
    for start in range(0, len(df_sql), BATCH_ROWS):
        batch = df_sql.iloc[start:start + BATCH_ROWS]
        batch.to_sql('customers', con=conn, if_exists='append', index=False, chunksize=10_000)
        print(f'  loaded {start + len(batch):,} / {len(df_sql):,} rows', flush=True)
print(f'Inserted {len(df_sql):,} rows into customers.')

  loaded 50,000 / 381,109 rows
  loaded 100,000 / 381,109 rows
  loaded 150,000 / 381,109 rows
  loaded 200,000 / 381,109 rows
  loaded 250,000 / 381,109 rows
  loaded 300,000 / 381,109 rows
  loaded 350,000 / 381,109 rows
  loaded 381,109 / 381,109 rows
Inserted 381,109 rows into customers.


## 8. Verify the insert

Cross-check the printed count against DBeaver: right-click `customers` -> View Data, or run `SELECT COUNT(*) FROM customers;` in the DBeaver SQL editor.

In [21]:
with engine.connect() as conn:
    row_count = conn.execute(text('SELECT COUNT(*) FROM customers')).scalar()

print(f'Rows in CSV:   {len(df):,}')
print(f'Rows in MySQL: {row_count:,}')
assert row_count == len(df), f'Row count mismatch: CSV has {len(df):,} rows, MySQL has {row_count:,}. Re-run the section 7 cell.'
print('Row counts match — import verified.')

Rows in CSV:   381,109
Rows in MySQL: 381,109
Row counts match — import verified.


## 9. Query the database

Reads data back out of MySQL with `pd.read_sql` to confirm the round-trip (CSV -> MySQL -> Python) works, and runs one aggregate query as an example. You can run the same two queries directly in DBeaver's SQL editor to see identical results.

In [22]:
sample = pd.read_sql('SELECT * FROM customers LIMIT 10', con=engine)
sample

,id,gender,age,driving_license,region_code,previously_insured,vehicle_age,vehicle_damage,annual_premium,policy_sales_channel,vintage,response
0,1,Male,44,1,28,0,> 2 Years,Yes,40454.0,26,217,1
1,2,Male,76,1,3,0,1-2 Year,No,33536.0,26,183,0
2,3,Male,47,1,28,0,> 2 Years,Yes,38294.0,26,27,1
3,4,Male,21,1,11,1,< 1 Year,No,28619.0,152,203,0
4,5,Female,29,1,41,1,< 1 Year,No,27496.0,152,39,0
5,6,Female,24,1,33,0,< 1 Year,Yes,2630.0,160,176,0
6,7,Male,23,1,11,0,< 1 Year,Yes,23367.0,152,249,0
7,8,Female,56,1,28,0,1-2 Year,Yes,32031.0,26,72,1
8,9,Female,24,1,3,1,< 1 Year,No,27619.0,152,28,0
9,10,Female,32,1,6,1,< 1 Year,No,28771.0,152,80,0


In [23]:
response_counts = pd.read_sql(
    'SELECT response, COUNT(*) AS num_customers FROM customers GROUP BY response',
    con=engine,
)
response_counts

,response,num_customers
0,0,334399
1,1,46710
